# Rotation curves：八组结果的参数指南（每次运行所选的一组）

**已支持八种组合：持续演化四组 + 冻结 halo 四组。** 下方章节的 Markdown 标题逐一写明参数和对应结果。
每次修改参数后，重新运行该章节的“参数”代码单元，然后运行“演化”和“绘图”单元；已有完整轨迹时可直接运行“绘图”。
默认 `nparticle = 600000`、`seed = 0`，演化至 11 Gyr。一次运行一个参数组合，不会自动运行全部八组。

This notebook contains the initialization, chunked orbit integration, diagnostics,
and MP4 rendering code. No rotation-workflow `.py` file is imported.
The project still requires AGAMA, NumPy, Matplotlib, imageio-ffmpeg and the stored
halo density/potential files; optional halo construction also uses `fdmtools.py`.

Run the shared definitions below once. In **Evolving halo**, change only
`halo_mass` (`1e9` or `1e10`) and `f` (`0.0` or `0.1`) to select the four
mass/scatter combinations. In **Frozen halo**, also set `freeze_time_gyr`
(default `1.0`). Each execution selects one experiment; repeat for the other
combinations. Here `f` is the fractional Gaussian scatter of the initial
tangential **speed magnitude**; the initial radial velocity is zero.

New integrations share the same seeded base positions and tangential directions
and remain in the halo-centered inertial frame. The original 0--1 Gyr halo ramp
is retained; freezing truncates the original history at the chosen snapshot.
Freeze time must match an existing halo snapshot and an orbit output time
(normally 0.1 Gyr spacing); no silent rounding is applied.

Compatible archived trajectories can be reused **read-only**. Legacy evolving
archives have tracer-centered stored positions and, for no scatter, nominal
times; their comparison with the new inertial-frame controls retains these
qualifications. Set `reuse_existing=False` to bypass legacy archives and produce
new matched trajectories. Completed notebook chunks remain restartable.

New outputs are saved under the selected halo's `s0/rotation_notebook/evolving/`
or `s0/rotation_notebook/freeze1.0Gyr/` directory. Filenames include particle
count, `vscatter`, seed and, when relevant, freeze time. The plot cell produces
the MP4, individual stills, and overlaid profiles at **0, 1, 5, 11 Gyr**.

# 0. 先运行 Shared definitions：共享代码，仅定义功能，不演化粒子
Run these definition cells once before either experiment section.

### Configuration and runtime

In [ ]:
import ctypes
import hashlib
import json
import os
import time
from pathlib import Path

os.environ.setdefault('OMP_NUM_THREADS', '12')
import numpy as np
for library in ('libgslcblas.so.0', 'libgsl.so.27'):
    candidate = Path.home()/'.local/opt/gsl/lib'/library
    if candidate.exists():
        ctypes.CDLL(str(candidate), mode=ctypes.RTLD_GLOBAL)
import agama as ag
ag.setUnits(mass=1, length=1, velocity=1)

HALO_MODELS = {
    1e9: 'ma1.0e-22M1.0e+09ts0.1rres512',
    1e10: 'ma1.0e-22M1.0e+10ts0.1rres256',
}
TIME_UNIT_GYR = 0.978  # Same conversion used by the archived halo timestamp tables.


def number_tag(value, minimum_decimals=1):
    text = np.format_float_positional(float(value), trim='-')
    if '.' not in text:
        text += '.'
    return text + '0'*max(0, minimum_decimals-len(text.split('.')[1]))


def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)


def make_rotation_config(halo_mass=1e10, f=0.1, freeze_time_gyr=None,
                         nparticle=600000, seed=0, end_time_gyr=11.,
                         output_dt_gyr=.1, chunk_size=100000, accuracy=1e-8,
                         root=None, halo_folder=None, reuse_existing=True,
                         output_dir=None, min_count=40):
    root = Path(root or Path.cwd()).resolve()
    mass = float(halo_mass)
    if mass not in HALO_MODELS:
        raise ValueError('Available archived halo masses are 1e9 and 1e10 solar masses.')
    if halo_folder is not None:
        known_mass = next((m for m,p in HALO_MODELS.items() if Path(halo_folder).name==p),None)
        if known_mass is not None and known_mass != mass:
            raise ValueError('halo_mass and halo_folder select different archived halos.')
    if not np.isfinite(f) or f < 0:
        raise ValueError('f must be a finite, non-negative fractional speed scatter.')
    if int(nparticle) != nparticle or nparticle <= 0 or int(chunk_size) != chunk_size or chunk_size <= 0:
        raise ValueError('Particle and chunk counts must be positive integers.')
    if not np.isfinite(accuracy) or accuracy <= 0:
        raise ValueError('accuracy must be finite and positive.')
    if not np.isfinite(output_dt_gyr) or output_dt_gyr <= 0 or not np.isfinite(end_time_gyr) or end_time_gyr <= 0:
        raise ValueError('Output cadence and end time must be finite and positive.')
    steps = int(round(end_time_gyr/output_dt_gyr))
    if not np.isclose(steps*output_dt_gyr, end_time_gyr, rtol=0, atol=1e-10):
        raise ValueError('End time must be an integer multiple of the output cadence.')
    if freeze_time_gyr is not None:
        freeze_time_gyr = float(freeze_time_gyr)
        if not np.isfinite(freeze_time_gyr) or not 0 <= freeze_time_gyr < end_time_gyr:
            raise ValueError('Freeze time must satisfy 0 <= freeze time < end time.')
        if not np.isclose(round(freeze_time_gyr/output_dt_gyr)*output_dt_gyr,
                          freeze_time_gyr, rtol=0, atol=1e-10):
            raise ValueError('Freeze time must coincide with an orbit output time.')
    halo = root/Path(halo_folder or HALO_MODELS[mass])/f's{int(seed)}'
    if not (halo/'pot_evolving_gradual.ini').exists():
        raise FileNotFoundError(f'Missing halo history: {halo}/pot_evolving_gradual.ini')
    history = 'evolving' if freeze_time_gyr is None else f'freeze{number_tag(freeze_time_gyr)}Gyr'
    tag = f'Np{int(nparticle)}_vscatter{number_tag(f,3)}'
    if freeze_time_gyr is not None:
        tag += f'_freeze{number_tag(freeze_time_gyr)}Gyr'
    tag += f'_s{int(seed)}'
    output = Path(output_dir) if output_dir is not None else halo/'rotation_notebook'/history
    params = dict(halo_mass=mass, f=float(f), freeze_time_gyr=freeze_time_gyr,
                  nparticle=int(nparticle), seed=int(seed), end_time_gyr=float(end_time_gyr),
                  output_dt_gyr=float(output_dt_gyr), chunk_size=int(chunk_size),
                  accuracy=float(accuracy), min_count=int(min_count),
                  coordinate_frame='Unshifted halo-centered inertial frame; no recentering',
                  implementation='rotation-notebook-v1')
    return dict(params, root=root, halo=halo, output=output, tag=tag,
                time_gyr=np.arange(steps+1)*output_dt_gyr,
                radius_edges=np.linspace(0,30,91), reuse_existing=bool(reuse_existing))

### Potential history and initial conditions

In [ ]:
def read_potential_table(config):
    table = []
    for line in (config['halo']/'pot_evolving_gradual.ini').read_text().splitlines():
        fields = line.split()
        if len(fields) != 2:
            continue
        try:
            timestamp = float(fields[0])
        except ValueError:
            continue
        table.append((timestamp, config['halo']/fields[1]))
    if not table or np.any(np.diff([row[0] for row in table]) <= 0):
        raise ValueError('Potential timestamps must be present and strictly increasing.')
    return table


def build_rotation_potentials(config):
    table = read_potential_table(config)
    selected = table
    fixed = None
    freeze_index = None
    if config['freeze_time_gyr'] is not None:
        epoch = config['freeze_time_gyr']
        matches = np.flatnonzero(np.isclose(np.array([r[0] for r in table])*TIME_UNIT_GYR,
                                           epoch, rtol=0, atol=1e-10))
        if not len(matches):
            raise ValueError(f'Freeze time {epoch} Gyr is not an archived halo snapshot; '
                             'the present histories have 0.1 Gyr spacing. No rounding is applied.')
        selected = table[:int(matches[0])+1]
        fixed = ag.Potential(str(selected[-1][1]))
        freeze_index = int(round(epoch/config['output_dt_gyr']))
    # Hash inputs so a changed halo cannot reuse trajectories from an older potential.
    digest = hashlib.sha256()
    for timestamp,path in selected:
        digest.update(str(timestamp).encode())
        digest.update(path.read_bytes())
    payload = {key:config[key] for key in ('halo_mass','f','freeze_time_gyr','nparticle',
        'seed','end_time_gyr','output_dt_gyr','chunk_size','accuracy','min_count','coordinate_frame','implementation')}
    payload['potential_sha256'] = digest.hexdigest()
    payload['halo_directory'] = str(config['halo'])
    manifest = config['output']/f'configuration_{config["tag"]}.json'
    if manifest.exists() and json.loads(manifest.read_text())['parameters'] != payload:
        fingerprint = hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:10]
        config['output'] = config['output']/f'run_{fingerprint}'
        manifest = config['output']/f'configuration_{config["tag"]}.json'
    config['output'].mkdir(parents=True, exist_ok=True)
    ini = config['output']/f'potential_{config["tag"]}.ini'
    ini.write_text('[Potential]\ntype=Evolving\nlinearInterp=True\nTimestamps\n'+
                   ''.join(f'{t:.17g} {path}\n' for t,path in selected))
    potential = ag.Potential(str(ini))
    points = np.array([[1.,.2,.3],[5.,2.,-1.],[10.,0.,0.],[20.,-3.,1.]])
    checks = dict(force_hold_max_abs_error=None, pre_freeze_match_max_abs_error=None)
    if fixed is not None:
        original = ag.Potential(str(config['halo']/'pot_evolving_gradual.ini'))
        hold_times = np.linspace(config['freeze_time_gyr'],config['end_time_gyr'],4)
        early_times = np.linspace(0,config['freeze_time_gyr'],4)
        hold = [np.max(np.abs(potential.force(points,t=t/TIME_UNIT_GYR)-fixed.force(points)))
                for t in hold_times]
        early = [np.max(np.abs(potential.force(points,t=t/TIME_UNIT_GYR)-
                              original.force(points,t=t/TIME_UNIT_GYR))) for t in early_times]
        if max(hold+early) > 1e-9:
            raise RuntimeError('Frozen potential failed force validation.')
        checks = dict(force_hold_max_abs_error=float(max(hold)),
                      pre_freeze_match_max_abs_error=float(max(early)))
    write_json(manifest,dict(parameters=payload,validation=checks))
    return dict(potential=potential, fixed=fixed, table=table, freeze_index=freeze_index)


def make_rotation_initial_conditions(config):
    n = config['nparticle']; seed = config['seed']
    rng = np.random.RandomState(seed)
    edges = np.linspace(0,30,601)
    counts = np.full(600,n//600); counts[:n%600] += 1
    radius = np.empty(n); start = 0
    for lo,hi,count in zip(edges[:-1],edges[1:],counts):
        radius[start:start+count] = (rng.random_sample(count)*(hi**3-lo**3)+lo**3)**(1/3)
        start += count
    rng.shuffle(radius)
    cosine = 2*rng.random_sample(n)-1
    sine = np.sqrt(np.maximum(1-cosine**2,0)); phi = 2*np.pi*rng.random_sample(n)
    position = np.column_stack([radius*sine*np.cos(phi),radius*sine*np.sin(phi),radius*cosine])
    baseline = ag.Potential(type='Multipole',density=ag.Density(str(config['halo']/'sph_harm_base_gradual.ini')))
    speed = np.sqrt(np.maximum(ag.G*np.asarray(baseline.enclosedMass(radius))/np.maximum(radius,1e-12),0))
    rhat = position/np.maximum(radius[:,None],1e-12)
    tangent = rng.normal(size=position.shape)
    tangent -= np.sum(tangent*rhat,axis=1,keepdims=True)*rhat
    tangent /= np.maximum(np.linalg.norm(tangent,axis=1)[:,None],1e-12)
    factors = np.maximum(1+np.random.default_rng(seed).normal(scale=config['f'],size=n),0)
    return np.column_stack([position,tangent*(speed*factors)[:,None]])

### Trajectory reuse and integration

In [ ]:
def validate_orbit_files(files,config,require_markers=False):
    if not files:
        return False
    total = 0
    for path in files:
        if require_markers and not path.with_suffix('.complete.json').exists():
            return False
        try:
            a = np.load(path,mmap_mode='r')
            if a.ndim != 3 or a.shape[:2] != (len(config['time_gyr']),6):
                return False
            if not np.all(np.isfinite(a[[0,-1]])) or np.any(np.all(a[-1]==0,axis=0)):
                return False
            total += a.shape[2]
        except (ValueError,OSError):
            return False
    return total == config['nparticle']


def existing_rotation_files(config):
    managed = sorted(config['output'].glob(f'xvta_{config["tag"]}_part*.npy'))
    if validate_orbit_files(managed,config,require_markers=True):
        return managed, 'Verified notebook trajectories; halo-centered coordinates and explicit times.'
    if not config['reuse_existing']:
        return None,None
    if config['freeze_time_gyr'] is None:
        tag = f'Np{config["nparticle"]}_vscatter{number_tag(config["f"],3)}'
        choices = [sorted(config['halo'].glob(f'xvta_{tag}_part*.npy'))]
        if config['f']==0:
            choices.append(sorted(config['halo'].glob(f'xvta_Np{config["nparticle"]}_part*.npy')))
        # Old runs only represent the documented seed-0, 11 Gyr, 0.1 Gyr setup.
        if config['seed']!=0 or config['end_time_gyr']!=11 or config['output_dt_gyr']!=.1 or config['accuracy']!=1e-8:
            return None,None
        for files in choices:
            if validate_orbit_files(files,config):
                return files, ('Legacy evolving archive, reused read-only: positions were tracer-centered; '
                               'velocities were not shifted. Legacy no-scatter times are nominal.')
    else:
        folder = config['halo']/f'freeze{number_tag(config["freeze_time_gyr"])}Gyr'
        old_config = folder/'configuration.json'
        if old_config.exists():
            old = json.loads(old_config.read_text())
            expected = {'halo_mass_label':config['halo_mass'],'nparticle':config['nparticle'],
                        'seed':config['seed'],'freeze_time_gyr':config['freeze_time_gyr'],
                        'end_time_gyr':config['end_time_gyr'],'output_dt_gyr':config['output_dt_gyr'],
                        'integration_accuracy':config['accuracy']}
            files = sorted(folder.glob(f'xvta_{config["tag"]}_part*.npy'))
            if (all(old.get(k)==v for k,v in expected.items()) and
                    config['f'] in old.get('scatter_fractions',[]) and
                    validate_orbit_files(files,config,require_markers=True)):
                return files, 'Verified frozen-control archive; halo-centered coordinates and explicit times.'
    return None,None


def evolve_rotation(config,render_only=False):
    state = build_rotation_potentials(config)
    files, provenance = existing_rotation_files(config)
    if files is None:
        if render_only:
            raise FileNotFoundError('No complete compatible trajectory set; run the evolution cell first.')
        ic = make_rotation_initial_conditions(config); files=[]
        for k,start in enumerate(range(0,config['nparticle'],config['chunk_size'])):
            subset=ic[start:start+config['chunk_size']]
            path=config['output']/f'xvta_{config["tag"]}_part{k:03d}.npy'
            done=path.with_suffix('.complete.json')
            small=dict(config,nparticle=len(subset))
            if not validate_orbit_files([path],small,require_markers=True):
                print(f'Integrating chunk {k+1}: {len(subset):,} particles',flush=True)
                begin=time.perf_counter()
                result=ag.orbit(potential=state['potential'],ic=subset,
                    time=config['end_time_gyr']/TIME_UNIT_GYR,timestart=0.,
                    trajsize=len(config['time_gyr']),accuracy=config['accuracy'],dtype='double')
                if not np.allclose(result[0,0]*TIME_UNIT_GYR,config['time_gyr'],rtol=0,atol=1e-10):
                    raise RuntimeError('AGAMA returned unexpected trajectory times.')
                a=np.lib.format.open_memmap(path,mode='w+',dtype='float32',
                                           shape=(len(config['time_gyr']),6,len(subset)))
                for j,orbit in enumerate(result):
                    a[:,:,j]=orbit[1]
                a.flush();del a,result
                write_json(done,dict(nparticle=len(subset),scatter=config['f'],
                                    elapsed_seconds=time.perf_counter()-begin))
            files.append(path)
            write_json(config['output']/f'status_{config["tag"]}.json',dict(stage='integrating',completed_chunks=k+1))
        provenance='New notebook trajectories; halo-centered coordinates and explicit times.'
    print(f'{config["halo_mass"]:.0e} solar masses, f={config["f"]}, freeze={config["freeze_time_gyr"]}: {provenance}')
    return dict(config=config,state=state,files=files,provenance=provenance)

### Full-population diagnostics

In [ ]:
def rotation_diagnostics(case):
    cfg=case['config']; state=case['state']
    path=cfg['output']/f'diagnostics_{cfg["tag"]}.npz'
    source_fingerprint = hashlib.sha256(json.dumps([
        (str(p),p.stat().st_size,p.stat().st_mtime_ns) for p in case['files']
    ]).encode()).hexdigest()
    if path.exists():
        cached = dict(np.load(path))
        if str(cached.get('source_fingerprint','')) == source_fingerprint:
            case['diagnostics']=cached;return cached
    maps=[np.load(p,mmap_mode='r') for p in case['files']]
    nt=len(cfg['time_gyr']); edges=cfg['radius_edges']; r=.5*(edges[:-1]+edges[1:])
    mean=np.full((nt,len(r)),np.nan); std=mean.copy(); phi=mean.copy(); vc=mean.copy()
    v2s=[]; forces=[]; rvs=[]; bv2=[]; bf=[]; brv=[]
    bound=[]; e0=[]; fi=state['freeze_index']; cache={}
    def energy(xv):
        return .5*np.sum(xv[:,3:]**2,axis=1)+state['fixed'].potential(xv[:,:3])
    if fi is not None:
        for a in maps:
            energies=energy(a[fi].T.astype(float));e0.append(energies);bound.append(energies<0)
    times=np.array([q[0]*TIME_UNIT_GYR for q in state['table']])
    def reference_mass(t):
        t=min(t,cfg['freeze_time_gyr']) if fi is not None else t
        hi=int(np.searchsorted(times,t,side='right'));hi=min(max(hi,1),len(times)-1)
        lo=hi-1; weight=float(np.clip((t-times[lo])/(times[hi]-times[lo]),0,1))
        for j in (lo,hi):
            if j not in cache:
                cache[j]=np.asarray(ag.Potential(str(state['table'][j][1])).enclosedMass(r))
        return (1-weight)*cache[lo]+weight*cache[hi]
    for ti,t in enumerate(cfg['time_gyr']):
        count=np.zeros(len(r)); total=count.copy(); total2=count.copy(); phitotal=count.copy()
        all_v2=all_force=all_rv=bound_v2=bound_force=bound_rv=0.
        for k,a in enumerate(maps):
            xv=a[ti].T.astype(float);x=xv[:,:3];v=xv[:,3:]
            radii=np.linalg.norm(x,axis=1); v2=np.sum(v*v,axis=1);rv=np.sum(x*v,axis=1)
            vt=np.linalg.norm(np.cross(x,v),axis=1)/np.maximum(radii,1e-12)
            R=np.linalg.norm(x[:,:2],axis=1)
            vphi=np.abs(x[:,0]*v[:,1]-x[:,1]*v[:,0])/np.maximum(R,1e-12)
            force=(state['fixed'].force(x) if fi is not None and ti>=fi
                   else state['potential'].force(x,t=t/TIME_UNIT_GYR))
            moment=-np.sum(x*force,axis=1)
            all_v2+=v2.sum();all_force+=moment.sum();all_rv+=rv.sum()
            if fi is not None:
                b=bound[k];bound_v2+=v2[b].sum();bound_force+=moment[b].sum();bound_rv+=rv[b].sum()
            idx=np.searchsorted(edges,radii,side='right')-1
            valid=(idx>=0)&(idx<len(r));ids=idx[valid]
            count+=np.bincount(ids,minlength=len(r))
            total+=np.bincount(ids,weights=vt[valid],minlength=len(r))
            total2+=np.bincount(ids,weights=vt[valid]**2,minlength=len(r))
            # The auxiliary azimuthal curve is binned in cylindrical radius R.
            iphi=np.searchsorted(edges,R,side='right')-1
            validphi=(iphi>=0)&(iphi<len(r))
            phitotal+=np.bincount(iphi[validphi],weights=vphi[validphi],minlength=len(r))
            if k==0: phicount=np.zeros(len(r))
            phicount+=np.bincount(iphi[validphi],minlength=len(r))
        good=count>=cfg['min_count'];mean[ti,good]=total[good]/count[good]
        std[ti,good]=np.sqrt(np.maximum(total2[good]/count[good]-mean[ti,good]**2,0))
        goodphi=phicount>=cfg['min_count'];phi[ti,goodphi]=phitotal[goodphi]/phicount[goodphi]
        vc[ti]=np.sqrt(np.maximum(ag.G*reference_mass(t)/r,0))
        v2s.append(all_v2/cfg['nparticle']);forces.append(all_force/cfg['nparticle']);rvs.append(all_rv/cfg['nparticle'])
        if fi is not None:
            nb=sum(b.sum() for b in bound)
            bv2.append(bound_v2/nb if nb else np.nan);bf.append(bound_force/nb if nb else np.nan);brv.append(bound_rv/nb if nb else np.nan)
        if ti%10==0: print(f'Diagnostics: t={t:g} Gyr',flush=True)
    data=dict(time_gyr=cfg['time_gyr'],radius=r,mean_vt=mean,std_vt=std,mean_abs_vphi=phi,vc=vc,
              source_fingerprint=source_fingerprint,
              mean_v2=np.array(v2s),mean_minus_r_dot_force=np.array(forces),mean_r_dot_v=np.array(rvs))
    if fi is not None:
        e0=np.concatenate(e0);ef=np.concatenate([energy(a[-1].T.astype(float)) for a in maps])
        error=np.abs(ef-e0)/np.maximum(np.abs(e0),1e-4*np.median(np.abs(e0)))
        data.update(bound_mean_v2=np.array(bv2),bound_mean_minus_r_dot_force=np.array(bf),
                    bound_mean_r_dot_v=np.array(brv),bound_fraction=sum(b.sum() for b in bound)/cfg['nparticle'],
                    energy_error_quantiles=np.quantile(error,[.5,.99,1]))
    np.savez(path,**data);case['diagnostics']=data
    return data

### Snapshots, virial diagnostics, and MP4 output

In [ ]:
def setup_rotation_plots():
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    for name in ('times.ttf','timesbd.ttf','timesi.ttf','timesbi.ttf'):
        path=Path('/mnt/c/Windows/Fonts')/name
        if path.exists():font_manager.fontManager.addfont(str(path))
    plt.rcParams.update({'font.family':'Times New Roman','font.size':11,
                         'mathtext.fontset':'stix','axes.grid':True,'grid.alpha':.15})
    return plt


def plot_rotation_snapshots(case, snapshot_times=(0.,1.,5.,11.)):
    plt=setup_rotation_plots();data=case.get('diagnostics') or rotation_diagnostics(case);cfg=case['config']
    fig,(ax,ratio)=plt.subplots(2,1,figsize=(8.5,7),sharex=True,gridspec_kw={'height_ratios':[1.4,1]})
    styles=[(':','o'),('--','s'),('-.','^'),('-',None)]
    actual=[]
    for t,(ls,marker) in zip(snapshot_times,styles):
        matches=np.flatnonzero(np.isclose(data['time_gyr'],t,atol=1e-10,rtol=0))
        if not len(matches):continue
        i=int(matches[0]);actual.append(float(t));draw=dict(color='black',ls=ls,marker=marker,
            markevery=9,ms=3.5,mfc='white',lw=1.5,label=f't = {t:g} Gyr')
        ax.plot(data['radius'],data['mean_vt'][i],**draw)
        ratio.plot(data['radius'],data['mean_vt'][i]/data['vc'][i],**draw)
        ax.plot(data['radius'],data['vc'][i],color='.6',ls=ls,lw=1,
                label=f'Circular speed, t = {t:g} Gyr')
    ax.set(ylabel='Mean tangential speed (km/s)',title=f'M = {cfg["halo_mass"]:.0e} solar masses; f = {cfg["f"]}; '+
           ('evolving halo' if cfg['freeze_time_gyr'] is None else f'halo frozen at {cfg["freeze_time_gyr"]:g} Gyr'))
    ax.legend(ncol=2,fontsize=8,frameon=False)
    ratio.axhline(1,color='.5',ls=':');ratio.set(xlim=(1,30),ylim=(.75,1.25),
        xlabel='Halo-centered spherical radius r (kpc)',ylabel=r'$\langle v_t\rangle/v_c$')
    if 'Legacy evolving' in case['provenance']:
        ratio.set_xlabel('Legacy tracer-centered spherical radius (kpc); nominal times')
    fig.tight_layout()
    stem=cfg['output']/f'rotation_snapshots_{cfg["tag"]}'
    fig.savefig(str(stem)+'.png',dpi=160);fig.savefig(str(stem)+'.pdf')
    plt.close(fig)
    return Path(str(stem)+'.png')


def summarize_rotation(case):
    data=case.get('diagnostics') or rotation_diagnostics(case);cfg=case['config'];s={}
    mask=(data['radius']>=5)&(data['radius']<=20)
    s['final_median_vt_over_vc_r5_20']=float(np.nanmedian((data['mean_vt'][-1]/data['vc'][-1])[mask]))
    s['provenance']=case['provenance']
    s['final_global_virial_ratio']=float(data['mean_v2'][-1]/data['mean_minus_r_dot_force'][-1])
    fi=case['state']['freeze_index']
    if fi is not None:
        t=data['time_gyr'][fi:]/TIME_UNIT_GYR;dt=np.diff(t)
        v2=data['bound_mean_v2'][fi:];w=data['bound_mean_minus_r_dot_force'][fi:]
        dv=np.sum(.5*(v2[1:]+v2[:-1])*dt);dw=np.sum(.5*(w[1:]+w[:-1])*dt)
        boundary=data['bound_mean_r_dot_v'][-1]-data['bound_mean_r_dot_v'][fi]
        s.update(final_bound_cumulative_virial_ratio=float(dv/dw),
                 boundary_corrected_cumulative_ratio=float((dv-boundary)/dw),
                 bound_fraction=float(data['bound_fraction']),
                 energy_relative_error_median_p99_max=data['energy_error_quantiles'].tolist())
        plt=setup_rotation_plots();fig,axes=plt.subplots(1,2,figsize=(10,3.6))
        axes[0].plot(data['time_gyr'],data['mean_v2']/data['mean_minus_r_dot_force'],color='black')
        axes[0].axvline(cfg['freeze_time_gyr'],color='.5',ls=':')
        numerator=np.cumsum(.5*(v2[1:]+v2[:-1])*dt);denominator=np.cumsum(.5*(w[1:]+w[:-1])*dt)
        axes[1].plot(data['time_gyr'][fi+1:],numerator/denominator,color='black')
        for ax in axes: ax.axhline(1,color='.5',ls=':');ax.set_xlabel('Time (Gyr)')
        axes[0].set_ylabel('Global virial ratio');axes[1].set_ylabel('Cumulative bound virial ratio')
        fig.tight_layout();fig.savefig(cfg['output']/f'virial_{cfg["tag"]}.png',dpi=150);plt.close(fig)
    if 'Legacy evolving' in case['provenance']:
        s['virial_warning']='Legacy moving-coordinate positions: do not interpret this virial ratio as an inertial-frame test.'
    s['note']='Finite-time virial balance and mean tangential speed equal to circular speed are different tests.'
    write_json(cfg['output']/f'summary_{cfg["tag"]}.json',s)
    return s


def render_rotation(case,frame_indices=None,fps=10,save_stills=True,decode=True):
    from matplotlib import animation
    import imageio_ffmpeg
    import subprocess
    plt=setup_rotation_plots();cfg=case['config'];data=case.get('diagnostics') or rotation_diagnostics(case)
    maps=[np.load(path,mmap_mode='r') for path in case['files']]
    fig,(ax,profile)=plt.subplots(2,1,figsize=(8.6,8.4),sharex=True)
    points=ax.scatter([],[],s=.5,alpha=.12,color='black',edgecolors='none',label='Displayed tracer sample')
    reference,=ax.plot(data['radius'],data['vc'][0],color='.5',ls=':',lw=2,label='Spherical circular speed')
    theory,=profile.plot(data['radius'],data['vc'][0],color='.5',ls=':',lw=2,label='Spherical circular speed')
    tangential,=profile.plot(data['radius'],data['mean_vt'][0],color='black',lw=1.6,label='Mean tangential speed')
    azimuthal,=profile.plot(data['radius'],data['mean_abs_vphi'][0],color='black',ls='--',
                             label='Mean azimuthal speed about z (binned in R)')
    band=[None]
    upper=max(float(np.nanmax(data['vc']))*1.5,float(np.nanmax(data['mean_vt']+data['std_vt']))*1.2)
    for axis in (ax,profile):axis.set(xlim=(0,30),ylim=(0,upper),ylabel='Speed (km/s)');axis.legend(fontsize=8)
    profile.set_xlabel('Spherical radius r / cylindrical radius R (kpc)')
    stride=max(1,int(np.ceil(cfg['nparticle']/80000)))
    def update(index):
        xv=np.concatenate([a[index,:,::stride].T for a in maps])[:80000].astype(float)
        x=xv[:,:3];v=xv[:,3:];r=np.linalg.norm(x,axis=1)
        vt=np.linalg.norm(np.cross(x,v),axis=1)/np.maximum(r,1e-12)
        points.set_offsets(np.column_stack([r,vt]));reference.set_ydata(data['vc'][index])
        theory.set_ydata(data['vc'][index]);tangential.set_ydata(data['mean_vt'][index])
        azimuthal.set_ydata(data['mean_abs_vphi'][index])
        if band[0] is not None:band[0].remove()
        band[0]=profile.fill_between(data['radius'],data['mean_vt'][index]-data['std_vt'][index],
                                    data['mean_vt'][index]+data['std_vt'][index],color='.7',alpha=.25)
        t=data['time_gyr'][index]
        state='evolving halo'
        if cfg['freeze_time_gyr'] is not None and t>=cfg['freeze_time_gyr']:
            state=f'halo frozen at {cfg["freeze_time_gyr"]:g} Gyr'
        ax.set_title(f'M={cfg["halo_mass"]:.0e} solar masses | f={cfg["f"]:g} | t={t:.1f} Gyr | {state}')
        if 'Legacy evolving' in case['provenance']:
            profile.set_xlabel('Legacy tracer-centered r / cylindrical R (kpc); nominal time')
        return points,reference,theory,tangential,azimuthal
    fig.tight_layout()
    if save_stills:
        for t in (0.,1.,5.,11.):
            matches=np.flatnonzero(np.isclose(data['time_gyr'],t,rtol=0,atol=1e-10))
            if len(matches):
                update(int(matches[0]));fig.savefig(cfg['output']/f'rotation_curve_{cfg["tag"]}_t{t:.1f}Gyr.png',dpi=150)
    frames=list(range(len(data['time_gyr']))) if frame_indices is None else list(frame_indices)
    if not frames or any(i<0 or i>=len(data['time_gyr']) for i in frames):
        raise ValueError('Animation frame indices are outside the trajectory.')
    destination=cfg['output']/f'vcirc_evolution_{cfg["tag"]}.mp4'
    plt.rcParams['animation.ffmpeg_path']=imageio_ffmpeg.get_ffmpeg_exe()
    movie=animation.FuncAnimation(fig,update,frames=frames,interval=1000/fps,blit=False)
    try:
        movie.save(destination,writer=animation.FFMpegWriter(fps=fps,codec='libx264',bitrate=1800,
                   extra_args=['-pix_fmt','yuv420p','-movflags','+faststart']),dpi=130)
    finally:plt.close(fig)
    if decode:
        subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-v','error','-xerror','-i',str(destination),
                        '-f','null','-'],check=True,capture_output=True)
    write_json(cfg['output']/f'status_{cfg["tag"]}.json',dict(stage='complete',mp4=str(destination),
        full_decode_verified=bool(decode),frame_count=len(frames)))
    print(f'Saved MP4: {destination}',flush=True)
    return destination

# 1. 持续演化四组 MP4：修改 `halo_mass` 和 `f`

在紧接下方的参数单元设置 `halo_mass` 和 `f`，其余参数保持默认：`nparticle = 600000`、`seed = 0`。
这里不冻结 halo；原有 0–1 Gyr ramp-up 后，继续使用随时间变化的 halo 势场，粒子演化至 11 Gyr。

### E1：`halo_mass = 1e9`，`f = 0.0` → $10^9\,M_\odot$，无 scatter，持续演化 MP4

### E2：`halo_mass = 1e9`，`f = 0.1` → $10^9\,M_\odot$，10% scatter，持续演化 MP4

### E3：`halo_mass = 1e10`，`f = 0.0` → $10^{10}\,M_\odot$，无 scatter，持续演化 MP4

### E4：`halo_mass = 1e10`，`f = 0.1` → $10^{10}\,M_\odot$，10% scatter，持续演化 MP4

## 持续演化参数：从 E1–E4 选择一组，修改下方 `halo_mass`、`f`

每组输出一个 `vcirc_evolution_Np600000_vscatter..._s0.mp4`，以及 t=0、1、5、11 Gyr 的叠加曲线。
质量自动选择对应 halo 目录（1e9 对应 rres512，1e10 对应 rres256），无需另外改绘图目录。

In [ ]:
halo_mass = 1e10              # Solar masses: 1e9 or 1e10
f = 0.1                       # Fractional speed scatter: 0.0 or 0.1
nparticle = 600000
seed = 0
reuse_existing = True         # False: bypass legacy trajectories; new chunks can still resume
build_halo = False            # Existing halo histories are normally reused

evolving_config = make_rotation_config(
    halo_mass=halo_mass, f=f, nparticle=nparticle, seed=seed,
    reuse_existing=reuse_existing,
)
print('Halo:', evolving_config['halo'])
print('Outputs:', evolving_config['output'])

## 可选 Build Halo：默认 `build_halo = False`，使用已有 halo
The original halo construction is retained below. The default `build_halo=False`
skips this expensive step and reads the existing halo selected above.
The builder writes its original `_curve` directory; it is independent of the
stored histories used here. To integrate a newly built history, pass its folder
through `halo_folder` when constructing the experiment configuration.

In [ ]:
if build_halo:
    import healpy as hp
    import fdmtools
    import os

    # import ctypes
    import numpy as np
    import matplotlib.pyplot as plt
    import scipy.interpolate as scinterp

    from tqdm import tqdm
    from multiprocessing import Pool, cpu_count
    import warnings
    # gsl_path="/XYFS01/pxy80/lib"
    # os.environ["LD_LIBRARY_PATH"] = gsl_path + ':' + os.environ.get('LD_LIBRARY_PATH','')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libintlc.so.5')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libirng.so')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libsvml.so')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libimf.so')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libgsl.so.27')
    # ctypes.cdll.LoadLibrary(gsl_path + '/libgslcblas.so.0') 

    Ggal = 4.30091727e-6
    hbar = 6.582119569509067e-16    # Reduced Planck constant in eV * s
    c_constant = 2.99792458e5       # Speed of light in km/s

    # Conversions:
    km_to_kpc = 3.240779289444365e-17       # Conversion from km to kpc
    km_s_to_kpc_gyr = 1.022712165045695     # Conversion from km/s to kpc/Gyr

    # Cosmology
    Om = 0.2865                     # Omega matter
    rho_crit = 133.36363195167573   # Critical background density (solMass / kpc^3)
    rho_m = rho_crit * Om           # Background matter density at present
    overdensity = 347.0             # Overdensity for spherical collapse
    rho_vir = overdensity * rho_m   # Average density within the virial radius

    # =============================
    # Runtime configuration
    # =============================
    have_soliton = True
    m_as = [1.0e-22]                                    # FDM particle mass in eV / c^2
    Nmaxes = [None]
    seeds = [0]                             # Realizations

    # Halo / profile setup
    r_setup_res = 512 if halo_mass == 1e9 else 256
    a = 10.
    M = halo_mass
    R_max = 160.
    R_fit = 50.

    # Time setup
    t_step = 0.1                                         # Gyr
    t_stop = 11.
    ramp_up = 10
    t_array = np.arange(start=0., stop=t_stop, step=t_step)
    timesteps = len(t_array)

    # Orbit IC setup (used in cell 2)
    nparticle = 600000
    velocity_scatter_frac = f

    t_to_Myr = 977.79222168
    warnings.filterwarnings('ignore')


    for Nmax in Nmaxes:
        for m_a in m_as:
            fdm = fdmtools.FDMTools(m_a)                          # Initialize FDM tools

            print(f'ma {m_a}')
            h_over_m = hbar / (m_a / c_constant**2) * km_to_kpc     # Scaled unit (h_bar / m_a) in astrophysical units: (kpc * km / s)

            # Define Base Potential
            r_setup = np.logspace(-3, 3, r_setup_res)

            rho_arr = M / (2. * np.pi * a**3) / (r_setup / a) / (1. + r_setup / a)**3

            alpha = 1./3.
            rc = 1.6 * (m_a / 1e-22)**-1 * (M / 1e9)**(-alpha)
            rhoc = 0.019 * (m_a / 1e-22)**-2 * rc**-4 / (1. + 0.091*(r_setup/rc)**2)**8 * 1e9
            transition = np.max(np.where(rhoc > rho_arr)) + 1

            rhoh = np.concatenate((rhoc[:transition], rho_arr[transition:]))
            if have_soliton:
                print(f'Using soliton core with rc {rc:.2f} kpc and rhoc {rhoc[0]:.2e} M_sun/kpc^3')
                rho_in = scinterp.CubicSpline(r_setup, rhoh)
            else:
                print('Using Hernquist profile without soliton core')
                rho_in = scinterp.CubicSpline(r_setup, rho_arr)

            #initialize potential
            phi, rho, M_tot = fdm.initialize_potential(rho_in, core=False, r_max=R_max)
            print(f'M_tot: {M_tot:.2e} M_sun')

            #invert the distribution function
            fE = fdm.DF_invert(phi, rho)

            print('Done.')

            folder = f'ma{m_a:.1e}M{M:.1e}ts{t_step}rres{r_setup_res}_curve' if have_soliton else f'ma{m_a:.1e}M{M:.1e}ts{t_step}rres{r_setup_res}_nosoliton_curve'
            if Nmax is not None:
                folder = f'{folder}_Nmax{Nmax}'


            for seed in seeds:
                print(f'seed {seed}')

                # Calculate eigenmodes and assign amplitudes
                E_nl, R_nl, a_nlm, phase_nlm, den, den_list = fdm.solve_amps_itr(
                    R_max, R_fit, phi, rho, M, fE,
                    maxiter=1, method='Isotropic', bins=50, schwarz_iter=2500, seed=seed
                )
                if Nmax is not None:
                    n_idx = np.arange(a_nlm.shape[0])[:, None]
                    l_idx = np.arange(a_nlm.shape[1])[None, :]
                    n_principle = n_idx + l_idx + 1
                    a_nlm[n_principle > Nmax] = 0.

                # Plot density profile after base-mode truncation (same spirit as solve_amps_itr diagnostics)
                r_profile = np.logspace(-2, np.log10(R_fit), 200)
                den_base_profile, _ = fdm.den_from_amps(r_profile, a_nlm, R_nl)
                plt.figure(figsize=(8, 6))
                plt.loglog(r_profile, rho_in(r_profile), 'k--', label='Target')
                plt.loglog(r_profile, den_base_profile, 'b-', label='Halo from truncated a_nlm')
                plt.xlim(0.01, R_fit)
                plt.xlabel(r'$r$ [kpc]')
                plt.ylabel(r'$\rho$ [$M_\odot$ / kpc$^3$]')
                plt.legend()
                plt.title(f'Nmax={Nmax}, seed={seed}')

                # Radius grid for reconstruction
                r_array = np.arange(start=0., stop=30., step=0.1)
                r_steps = 251
                r_array[0] = 0.00001

                # Export to Agama format

                prefix = f'{folder}/s{seed}'
                if not os.path.exists(prefix):
                    os.makedirs(prefix)
                nmax_tag = 'all' if Nmax is None else str(Nmax)
                plt.savefig(f'{prefix}/density_profile_n{nmax_tag}.png', dpi=150)
                plt.close()

                shape = np.shape(a_nlm)
                nmax_eff = shape[0]
                valid_nl = E_nl != 0.
                l_values = np.arange(shape[1])
                total_modes = int(np.sum(valid_nl * (2 * l_values[None, :] + 1)))
                active_nl = valid_nl & (a_nlm != 0.)
                active_modes = int(np.sum(active_nl * (2 * l_values[None, :] + 1)))
                print(f"Seed {seed}: total modes={total_modes}, active modes={active_modes}")

                for ti in tqdm(range(timesteps)):

                    timestep = t_array[ti]
                    NSIDE = 12
                    NPIX = hp.nside2npix(NSIDE)
                    rho_lms_size = hp.Alm.getsize(3 * NSIDE - 1)

                    grid = np.zeros((r_steps, NPIX))
                    rho_lms = np.zeros((r_steps, rho_lms_size), dtype=complex)

                    lmax = shape[1] - 1  # 原来代码没有

                    for i in range(r_steps):

                        a_nlm_real = np.zeros((lmax+2)*(lmax+1)//2, dtype=complex)
                        a_nlm_imag = np.zeros((lmax+2)*(lmax+1)//2, dtype=complex)

                        for l in range(shape[1]):
                            for m in range(0, l+1):
                                AR = 0.
                                idx = (2*lmax+3-m)*m//2 + (l-m)
                                for n in range(nmax_eff):
                                    if E_nl[n, l] == 0.:
                                        continue
                                    AR = a_nlm[n, l] * R_nl[n, l](r_array[i])
                                    phase_new = -E_nl[n, l] / h_over_m * km_s_to_kpc_gyr * timestep
                                    alm = np.cos(phase_nlm[n, l, m] + phase_new) + 1.j * np.sin(phase_nlm[n, l, m] + phase_new)
                                    almm = (np.cos(phase_nlm[n, l, -m] + phase_new) - 1.j * np.sin(phase_nlm[n, l, -m] + phase_new)) * (-1.)**m
                                    if m == 0:
                                        a_nlm_real[idx] += AR * alm
                                        a_nlm_imag[idx] += AR * alm / (1.j)
                                    else:
                                        a_nlm_real[idx] += AR / 2. * (alm + almm)
                                        a_nlm_imag[idx] += AR / 2.j * (alm - almm)

                        new_map_real = hp.alm2map(a_nlm_real, nside=NSIDE)
                        new_map_imag = hp.alm2map(a_nlm_imag, nside=NSIDE) * 1.j
                        new_map = new_map_real + new_map_imag

                        if ti < ramp_up:
                            rat = ti / ramp_up
                        else:
                            rat = 1.

                        grid[i] = rat * np.abs(new_map)**2
                        rho_lms[i] = hp.map2alm(grid[i])

                    lmax = hp.Alm.getlmax(rho_lms.shape[1])

                    f = open(f"{prefix}/sph_harm_gradual"+str(ti).zfill(3)+".ini", "w")
                    f.write('[Density]\n')
                    f.write('type=DensitySphericalHarmonic\n')
                    f.write('gridSizeR='+str(r_steps)+'\n')
                    f.write('lmax='+str(lmax)+'\n')
                    f.write('Coefficients\n')
                    f.write('#rho\n')
                    f.write(f'{"#radius": <17}')
                    for l in range(lmax+1):
                        for m in range(-l,l+1):
                            f.write(f'{"l="+str(l)+",m="+str(m): <17}')
                    f.write('\n')

                    for i in range(r_steps):
                        f.write(f"{r_array[i]:{16}.{10}}")
                        for l in range(lmax+1):
                            for m in range(-l,l+1):
                                if m>0:
                                    idx = (2*lmax+3-m)*m//2 + (l-m)
                                    rho_lm = np.sqrt(2) * rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                                elif m==0:
                                    idx = (2*lmax+3-m)*m//2 + (l-m)
                                    rho_lm = rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                                elif m<0:
                                    idx = (2*lmax+3+m)*(-m)//2 + (l+m)
                                    alm_p = rho_lms[i,idx]
                                    alm_m = rho_lms[i,idx].conjugate()
                                    rho_lm = -np.sqrt(2) * rho_lms[i,idx].imag / (np.sqrt(4. * np.pi))
                                f.write(' '+f"{rho_lm:{16}.{10}}")
                        f.write('\n')

                    f.close()

                dens = np.zeros(r_steps)
                shape = np.shape(a_nlm)
                for l in range(shape[1]):
                    for n in range(nmax_eff):
                        if E_nl[n, l] == 0.:
                            continue
                        dens += a_nlm[n, l]**2 * R_nl[n, l](r_array[0:r_steps])**2 * (2 * l + 1) / np.pi / 4.

                NSIDE = 3
                NPIX = hp.nside2npix(NSIDE)
                rho_lms_size = hp.Alm.getsize(3 * NSIDE - 1)

                grid = np.zeros((r_steps, NPIX))
                rho_lms = np.zeros((r_steps, rho_lms_size), dtype=complex)

                for i in range(r_steps):
                    grid[i] = dens[i]
                    rho_lms[i] = hp.map2alm(grid[i])

                lmax = hp.Alm.getlmax(rho_lms.shape[1])

                f = open(f"{prefix}/sph_harm_base_gradual.ini", "w")
                f.write('[Density]\n')
                f.write('type=DensitySphericalHarmonic\n')
                f.write('gridSizeR='+str(r_steps)+'\n')
                f.write('lmax='+str(lmax)+'\n')
                f.write('Coefficients\n')
                f.write('#rho\n')
                f.write(f'{"#radius": <17}')
                for l in range(lmax+1):
                    for m in range(-l,l+1):
                        f.write(f'{"l="+str(l)+",m="+str(m): <17}')
                f.write('\n')

                for i in range(r_steps):
                    f.write(f"{r_array[i]:{16}.{10}}")
                    for l in range(lmax+1):
                        for m in range(-l,l+1):
                            if m>0:
                                idx = (2*lmax+3-m)*m//2 + (l-m)
                                rho_lm = np.sqrt(2) * rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                            elif m==0:
                                idx = (2*lmax+3-m)*m//2 + (l-m)
                                rho_lm = rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                            elif m<0:
                                idx = (2*lmax+3+m)*(-m)//2 + (l+m)
                                alm_p = rho_lms[i,idx]
                                alm_m = rho_lms[i,idx].conjugate()
                                rho_lm = -np.sqrt(2) * rho_lms[i,idx].imag / (np.sqrt(4. * np.pi))
                            f.write(' '+f"{rho_lm:{16}.{10}}")
                    f.write('\n')

                f.close()

                # Build gradual base
                for ti in range(ramp_up+1):

                    rat = 1. - ti / ramp_up

                    grid = np.zeros((r_steps, NPIX))
                    rho_lms = np.zeros((r_steps, rho_lms_size), dtype=complex)

                    for i in range(r_steps):
                        grid[i] = dens[i] * rat
                        rho_lms[i] = hp.map2alm(grid[i])

                    lmax = hp.Alm.getlmax(rho_lms.shape[1])

                    f = open(f"{prefix}/sph_harm_base_gradual"+str(ti).zfill(3)+".ini", "w")
                    f.write('[Density]\n')
                    f.write('type=DensitySphericalHarmonic\n')
                    f.write('gridSizeR='+str(r_steps)+'\n')
                    f.write('lmax='+str(lmax)+'\n')
                    f.write('Coefficients\n')
                    f.write('#rho\n')
                    f.write(f'{"#radius": <17}')
                    for l in range(lmax+1):
                        for m in range(-l,l+1):
                            f.write(f'{"l="+str(l)+",m="+str(m): <17}')
                    f.write('\n')

                    for i in range(r_steps):
                        f.write(f"{r_array[i]:{16}.{10}}")
                        for l in range(lmax+1):
                            for m in range(-l,l+1):
                                if m>0:
                                    idx = (2*lmax+3-m)*m//2 + (l-m)
                                    rho_lm = np.sqrt(2) * rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                                elif m==0:
                                    idx = (2*lmax+3-m)*m//2 + (l-m)
                                    rho_lm = rho_lms[i,idx].real / (np.sqrt(4. * np.pi))
                                elif m<0:
                                    idx = (2*lmax+3+m)*(-m)//2 + (l+m)
                                    alm_p = rho_lms[i,idx]
                                    alm_m = rho_lms[i,idx].conjugate()
                                    rho_lm = -np.sqrt(2) * rho_lms[i,idx].imag / (np.sqrt(4. * np.pi))
                                f.write(' '+f"{rho_lm:{16}.{10}}")
                        f.write('\n')

                    f.close()

            print('Done.')

            import agama as ag

            ag.setUnits(mass=1, length=1, velocity=1)

            def process_snapshot(args):
                i, prefix = args
                global ramp_up

                if i < ramp_up:
                    den1 = ag.Density(f'{prefix}/sph_harm_gradual{i:03d}.ini')
                    den2 = ag.Density(f'{prefix}/sph_harm_base_gradual{i:03d}.ini')
                    den = ag.Density(den1, den2)
                else:
                    den = ag.Density(f'{prefix}/sph_harm_gradual{i:03d}.ini')

                pot = ag.Potential(type='multipole', density=den, lmax=20, rmin=0.0001, rmax=30)
                filename = f'{prefix}/sph_harm_pot_gradual{i:03d}.ini'
                pot.export(filename)

                return i, filename

            numsnaps = len(t_array)
            snapstep = t_step / 0.978

            for seed in seeds:
                prefix = f'{folder}/s{seed}'

                with open(f'{prefix}/pot_evolving_gradual.ini', 'w') as masterfile:
                    masterfile.write('''[Potential]
            type=Evolving
            linearInterp=True
            Timestamps
            ''')

                    args_list = [(i, prefix) for i in range(numsnaps)]

                    num_workers = min(cpu_count(), numsnaps)

                    with Pool(processes=num_workers) as pool:
                        results = []
                        for result in tqdm(
                            pool.imap_unordered(process_snapshot, args_list),
                            total=numsnaps,
                            desc=f'Seed {seed} potentials'
                        ):
                            results.append(result)

                    results.sort(key=lambda x: x[0])

                    for i, filename in results:
                        masterfile.write(f'{snapstep * i} sph_harm_pot_gradual{i:03d}.ini\n')
else:
    print('Using the existing halo; construction skipped.')

## 持续演化：按上方 `halo_mass`、`f` 运行所选 E1–E4（默认至 11 Gyr）
Reuse a complete compatible archive or integrate the selected case in restartable chunks.

In [ ]:
evolving_case = evolve_rotation(evolving_config)

## 持续演化输出：所选 E1–E4 的 MP4，以及 t=0、1、5、11 Gyr 曲线
The curves use all particles. The movie displays at most 80,000 individual
tracers, includes the initial frame, and uses H.264/yuv420p with faststart.
The auxiliary azimuthal speed is about z, $|v_\phi|=|L_z|/R$, binned in
cylindrical radius R; it is not a line-of-sight velocity.

In [ ]:
evolving_case = evolve_rotation(evolving_config, render_only=True)
rotation_diagnostics(evolving_case)
evolving_profiles = plot_rotation_snapshots(evolving_case, snapshot_times=(0, 1, 5, 11))
evolving_summary = summarize_rotation(evolving_case)
evolving_mp4 = render_rotation(evolving_case)
from IPython.display import display, Image, Video
display(Image(filename=str(evolving_profiles)))
display(Video(filename=str(evolving_mp4), embed=True))
print(json.dumps(evolving_summary, indent=2))
print('MP4:', evolving_mp4)

# 2. 冻结 halo 四组 MP4：修改 `halo_mass`、`f`、`freeze_time_gyr`（默认 1.0 Gyr）

先运行共享定义；无需先运行持续演化章节。在下方参数单元选择质量和 scatter。
以下 F1–F4 固定 `freeze_time_gyr = 1.0`：保留前 0–1 Gyr 的原有 ramp-up，
随后固定 1 Gyr 的完整非球对称 halo 势场，仅继续演化粒子至 11 Gyr。

### F1：`halo_mass = 1e9`，`f = 0.0`，`freeze_time_gyr = 1.0` → $10^9\,M_\odot$，无 scatter，1 Gyr 冻结 MP4

### F2：`halo_mass = 1e9`，`f = 0.1`，`freeze_time_gyr = 1.0` → $10^9\,M_\odot$，10% scatter，1 Gyr 冻结 MP4

### F3：`halo_mass = 1e10`，`f = 0.0`，`freeze_time_gyr = 1.0` → $10^{10}\,M_\odot$，无 scatter，1 Gyr 冻结 MP4

### F4：`halo_mass = 1e10`，`f = 0.1`，`freeze_time_gyr = 1.0` → $10^{10}\,M_\odot$，10% scatter，1 Gyr 冻结 MP4

## 冻结参数：从 F1–F4 选择一组，修改下方 `halo_mass`、`f`，保留 `freeze_time_gyr = 1.0`

若要改变冻结时刻，例如 2 Gyr，则设置 `freeze_time_gyr = 2.0`，其余流程相同。
冻结时刻须对应已有 halo 快照及轨道输出时刻（默认每 0.1 Gyr）；程序会检查，不会自动取整。
每组输出一个含 `vscatter..._freeze1.0Gyr_s0` 的 MP4，以及 t=0、1、5、11 Gyr 的叠加曲线。
这些是四个实验组合；每个组合又包含四个时刻的曲线。整体维里平衡不要求每个半径的平均切向速度等于圆周速度。

In [ ]:
halo_mass = 1e10              # Solar masses: 1e9 or 1e10
f = 0.1                       # Fractional speed scatter: 0.0 or 0.1
freeze_time_gyr = 1.0          # Must coincide with a stored halo snapshot
nparticle = 600000
seed = 0
reuse_existing = True

frozen_config = make_rotation_config(
    halo_mass=halo_mass, f=f, freeze_time_gyr=freeze_time_gyr,
    nparticle=nparticle, seed=seed, reuse_existing=reuse_existing,
)
print('Halo:', frozen_config['halo'])
print('Outputs:', frozen_config['output'])

## 冻结演化：按上方 `halo_mass`、`f` 运行所选 F1–F4，在 `freeze_time_gyr` 冻结 halo

In [ ]:
frozen_case = evolve_rotation(frozen_config)

## 冻结输出：所选 F1–F4 的 MP4，以及 t=0、1、5、11 Gyr 曲线和维里诊断
Energy changes are measured from the selected freeze epoch to the final time.
All force and virial diagnostics use the actual nonspherical force at all radii.
The finite-time boundary-corrected virial identity is a consistency check,
not an independent demonstration of stationarity. The archived lower-mass
controls have outer-orbit energy-error tails; inspect the reported quantiles.

In [ ]:
frozen_case = evolve_rotation(frozen_config, render_only=True)
rotation_diagnostics(frozen_case)
frozen_profiles = plot_rotation_snapshots(frozen_case, snapshot_times=(0, 1, 5, 11))
frozen_summary = summarize_rotation(frozen_case)
frozen_mp4 = render_rotation(frozen_case)
from IPython.display import display, Image, Video
display(Image(filename=str(frozen_profiles)))
display(Video(filename=str(frozen_mp4), embed=True))
print(json.dumps(frozen_summary, indent=2))
print('MP4:', frozen_mp4)